# Asset Class Trend Following 策略回測 (2019-2024)

本筆記本實作了基於 SMA 與 ROC 的動能策略，並執行全週期 (2019-2024) 回測。
參數：SMA 100, ROC 10, MA Stop 15

本程式包含完整的資料處理、策略回測以及詳細 Excel 報表產出邏輯。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xlsxwriter

import warnings
warnings.filterwarnings('ignore')

In [ ]:

def load_data(filepath):
    df = pd.read_excel(filepath, header=[0, 1])
    # 第二欄為日期
    date_col = df.columns[1]
    df.set_index(date_col, inplace=True)
    df.index = pd.to_datetime(df.index)
    # 移除首欄標籤
    df.drop(df.columns[0], axis=1, inplace=True)
    # 缺失值處理：ffill 與 bfill
    df = df.ffill().bfill()
    return df

df = load_data('個股1.xlsx')
print(f"資料讀取完成，標的數: {df.shape[1]}, 天數: {df.shape[0]}")


In [ ]:

def run_backtest(df, sma_len, roc_len, stop_loss_type, stop_loss_val, rb_period=5, rb_offset=0, initial_capital=30_000_000):
    prices = df.copy()
    tickers = prices.columns.get_level_values(0)
    names = prices.columns.get_level_values(1)
    ticker_to_name = dict(zip(tickers, names))
    prices.columns = tickers

    sma = prices.rolling(int(sma_len)).mean()
    roc = prices.pct_change(int(roc_len))

    dates = prices.index
    n_days = len(dates)

    cash = initial_capital
    holdings = {} # ticker -> {'shares': float, 'max_price': float, 'entry_date': date, 'entry_price': float}
    equity = pd.Series(index=dates, dtype=float)
    trade_log = []
    holdings_log = []
    pending_trades = []

    start_idx = max(int(sma_len), int(roc_len))
    if start_idx >= n_days:
        return pd.Series([initial_capital]*n_days, index=dates), [], []

    for i in range(start_idx, n_days):
        curr_date = dates[i]
        curr_prices = prices.iloc[i]

        if pending_trades:
            sells = [t for t in pending_trades if t['type'] == 'sell']
            buys = [t for t in pending_trades if t['type'] == 'buy']
            for trade in sells:
                ticker = trade['ticker']
                p = curr_prices[ticker]
                cash += trade['shares'] * p
                entry_info = holdings.pop(ticker, None)
                if entry_info:
                    trade_log.append({
                        'Date': curr_date, 'Ticker': ticker, 'Name': ticker_to_name[ticker],
                        'Type': 'Sell', 'Price': p, 'Shares': trade['shares'],
                        'Reason': trade['reason'], 'Entry Date': entry_info['entry_date'],
                        'Entry Price': entry_info['entry_price'],
                        'Return': (p / entry_info['entry_price']) - 1 if entry_info['entry_price'] != 0 else 0
                    })
            for trade in buys:
                ticker = trade['ticker']
                p = curr_prices[ticker]
                if p > 0:
                    amount = min(trade['amount'], cash)
                    if amount > 0:
                        shares = amount / p
                        holdings[ticker] = {'shares': shares, 'max_price': p, 'entry_date': curr_date, 'entry_price': p}
                        cash -= amount
                        trade_log.append({
                            'Date': curr_date, 'Ticker': ticker, 'Name': ticker_to_name[ticker],
                            'Type': 'Buy', 'Price': p, 'Shares': shares,
                            'Reason': trade['reason'], 'Momentum_Value': trade.get('momentum', 0)
                        })
            pending_trades = []

        port_value = cash
        for ticker, info in holdings.items():
            port_value += info['shares'] * curr_prices[ticker]
            holdings[ticker]['max_price'] = max(holdings[ticker]['max_price'], curr_prices[ticker])
        equity.iloc[i] = port_value
        holdings_log.append({'Date': curr_date, 'Holdings': {t: info['shares'] for t, info in holdings.items()}, 'Equity': port_value})

        if i == n_days - 1: continue

        for ticker in list(holdings.keys()):
            info = holdings[ticker]
            if stop_loss_type == 'ma':
                ma_days = int(stop_loss_val)
                if i >= ma_days:
                    ma_val = prices[ticker].rolling(ma_days).mean().iloc[i]
                    if curr_prices[ticker] < ma_val:
                        pending_trades.append({'ticker': ticker, 'type': 'sell', 'shares': info['shares'], 'reason': f'MA Stop ({ma_days})'})

        if (i - start_idx) % rb_period == rb_offset:
            eligible = (prices.iloc[i] > sma.iloc[i]) & (roc.iloc[i] > 0)
            top_3 = roc.iloc[i][eligible].sort_values(ascending=False).head(3).index.tolist()
            for t in list(holdings.keys()):
                if t not in top_3 and not any(pt['ticker'] == t and pt['type'] == 'sell' for pt in pending_trades):
                    pending_trades.append({'ticker': t, 'type': 'sell', 'shares': holdings[t]['shares'], 'reason': '跌出排名'})
            to_buy = [t for t in top_3 if t not in holdings and not any(pt['ticker'] == t and pt['type'] == 'buy' for pt in pending_trades)]
            if to_buy:
                filled_slots = len([t for t in top_3 if t in holdings and not any(pt['ticker'] == t and pt['type'] == 'sell' for pt in pending_trades)])
                open_slots = 3 - filled_slots
                if open_slots > 0:
                    amount_per_slot = port_value / 3
                    for t in to_buy[:open_slots]:
                        pending_trades.append({'ticker': t, 'type': 'buy', 'amount': amount_per_slot, 'reason': f'ROC 排名進入前 3 ({roc.iloc[i][t]:.4f})', 'momentum': roc.iloc[i][t]})

    return equity.ffill().fillna(initial_capital), trade_log, holdings_log


In [ ]:

def calculate_metrics(equity):
    tr = (equity.iloc[-1] / equity.iloc[0]) - 1
    years = (equity.index[-1] - equity.index[0]).days / 365.25
    cagr = (1 + tr) ** (1 / years) - 1
    mdd = ((equity / equity.cummax()) - 1).min()
    daily_ret = equity.pct_change().dropna()
    win_rate = (daily_ret > 0).mean()
    return {'CAGR': cagr, 'MaxDD': mdd, 'Calmar': cagr / abs(mdd), 'WinRate': win_rate, 'TotalReturn': tr}


In [ ]:

# 執行回測
params = (100, 10, 'ma', 15)
equity, trades, h_log = run_backtest(df, *params)
metrics = calculate_metrics(equity)

# 準備 Trades 工作表
trades_df = pd.DataFrame(trades)
if not trades_df.empty:
    trades_df['SMA_Param'] = params[0]
    trades_df['ROC_Param'] = params[1]
    trades_df['StopLoss_Param'] = f"{params[2]}_{params[3]}"
    def get_desc(row):
        if row['Type'] == 'Buy':
            return f"選取動能值(ROC)為 {row['Momentum_Value']:.4f} 的資產 {row['Name']} 進場"
        else:
            return f"資產 {row['Name']} 因 {row['Reason']} 出場"
    trades_df['繁體中文說明'] = trades_df.apply(get_desc, axis=1)

# 準備 Equity_Curve 工作表
equity_df = pd.DataFrame(equity, columns=['Equity'])
equity_df['Drawdown'] = (equity_df['Equity'] / equity_df['Equity'].cummax()) - 1

# 準備 Equity_Hold 工作表
h_rows = []
for entry in h_log:
    d = entry['Date']
    for t, s in entry['Holdings'].items():
        h_rows.append({'Date': d, 'Ticker': t, 'Shares': s})
holdings_df = pd.DataFrame(h_rows)

# 準備 Summary 工作表
summary_data = {
    'Metric': ['CAGR', 'MaxDD', 'Calmar', 'WinRate', 'TotalReturn', 'SMA', 'ROC', 'SL_Type', 'SL_Val'],
    'Value': [
        f"{metrics['CAGR']*100:.2f}%",
        f"{metrics['MaxDD']*100:.2f}%",
        round(metrics['Calmar'], 2),
        f"{metrics['WinRate']*100:.2f}%",
        f"{metrics['TotalReturn']*100:.2f}%",
        params[0], params[1], params[2], params[3]
    ]
}
summary_df = pd.DataFrame(summary_data)

# 匯出至 Excel
filename = 'trendresults_20192024.xlsx'
with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
    trades_df.to_excel(writer, sheet_name='Trades', index=False)
    equity_df.to_excel(writer, sheet_name='Equity_Curve')
    holdings_df.to_excel(writer, sheet_name='Equity_Hold', index=False)
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
print(f"Excel 報表已產生: {filename}")


In [ ]:

plt.figure(figsize=(12, 6))
plt.plot(equity, label='Equity Curve')
plt.title('Strategy Performance (2019-2024)')
plt.legend()
plt.show()
for k, v in metrics.items(): print(f"{k}: {v:.4f}")
